# 📍 Google Maps High-Throughput Scraper & Media Pipeline
### Complete Workflow for Google Colab / Kaggle with Google Drive Storage

This notebook runs the automated Google Maps data extraction and high-resolution photo pipeline.

**Key Capabilities:**
- 🔄 **100% Resumable & Fault-Tolerant:** Saves state to SQLite after every batch. If Colab disconnects or stops, re-running the cells continues exactly where it left off without duplicate queries or lost credits.
- ☁️ **Full Google Drive Integration:** Loads source databases (`cafes_cafe_shard00.db`) and saves output databases (`google_maps.db`), raw JSON archives (`google_maps_raw.jsonl`), and photos (`downloaded_images/`) directly on Google Drive.
- 🔑 **Multi-Key Auto-Rotation:** Automatically cycles through multiple Apify API tokens, monitors USD credit balances in real time, and fails over smoothly when quotas are exhausted.
- 📸 **Original Quality WebP Media:** Downloads camera-original resolution photos (`=s0-rw`) concurrently with periodic progress commits.

## 1. Mount Google Drive & Environment Setup
Mount your Google Drive to load source databases and persist all output data.

In [ ]:
import os
import sys

# Check runtime environment
IS_COLAB = 'google.colab' in sys.modules
IS_KAGGLE = 'kaggle_web_client' in sys.modules or os.path.exists('/kaggle/working')

if IS_COLAB:
    from google.colab import drive
    drive.mount('/content/drive')
    print("✅ Google Drive successfully mounted at /content/drive")
elif IS_KAGGLE:
    print("ℹ️ Running in Kaggle environment. Outputs will be saved to /kaggle/working")
else:
    print("ℹ️ Running in Local / Standard Python environment")

## 2. Install Dependencies & Navigate to Project Directory
Install the required libraries (`apify-client`, `requests`, `tqdm`). If you have cloned the repository in Colab, navigate into the project directory.

In [ ]:
# If running in Colab from scratch, uncomment below to clone your repository:
# !git clone https://github.com/osumy/apify_maps.git
# %cd apify_maps

!pip install -q -r requirements.txt
print("✅ Dependencies installed successfully.")

## 3. Configuration & Paths Setup
Configure your Google Drive paths and Apify API tokens.
> **Tip:** You can provide multiple Apify tokens separated by commas. The system will automatically rotate between them.

In [ ]:
import os

# ---------------------------------------------------------------------------
# 1. Apify API Token(s)
# You can also use Colab Secrets (Left sidebar > Key icon) with name 'APIFY_TOKEN'
# ---------------------------------------------------------------------------
colab_token = None
try:
    from google.colab import userdata
    colab_token = userdata.get('APIFY_TOKEN')
except Exception:
    pass

APIFY_TOKENS = colab_token or [
    "apify_api_eJGiVT4jV2uTq623XG12n8Y1uX8q2wP8tx",
    "apify_api_o0Y2kx8j03Yh1Nf98Gj602pBq013NBzco",
    "apify_api_HLftjk3l92uGq633jG10n8Y0uX724K5oM"
]

# ---------------------------------------------------------------------------
# 2. Google Drive Storage Directory
# Set where you want your database, jsonl, and photos stored
# ---------------------------------------------------------------------------
if IS_COLAB:
    DRIVE_DIR = "/content/drive/MyDrive/apify_maps_data"
elif IS_KAGGLE:
    DRIVE_DIR = "/kaggle/working/apify_maps_data"
else:
    DRIVE_DIR = os.path.abspath("./data")

os.makedirs(DRIVE_DIR, exist_ok=True)

# Source database containing initial venues (e.g. cafes_cafe_shard00.db)
# Upload your source database to this Google Drive folder or adjust the path
SOURCE_DB_PATH = os.path.join(DRIVE_DIR, "cafes_cafe_shard00.db")

# Output paths on Google Drive
OUTPUT_DB_PATH = os.path.join(DRIVE_DIR, "google_maps.db")
RAW_JSONL_PATH = os.path.join(DRIVE_DIR, "google_maps_raw.jsonl")
IMAGES_DIR = os.path.join(DRIVE_DIR, "downloaded_images")

# Operational Parameters
BATCH_SIZE = 50           # Venues per Apify Actor run
MAX_IMAGE_WORKERS = 25    # Concurrent photo download threads
MAX_IMAGES_PER_PLACE = 50 # Photos requested per venue
MAX_REVIEWS_PER_PLACE = 15 # Reviews requested per venue

print("Project Paths Configured:")
print(f"  • Source DB:    {SOURCE_DB_PATH}")
print(f"  • Output DB:    {OUTPUT_DB_PATH}")
print(f"  • Raw JSONL:    {RAW_JSONL_PATH}")
print(f"  • Photos Dir:   {IMAGES_DIR}")

## 4. Key Pool Balances & Health Check
Inspect all configured Apify tokens, their real-time USD credit balances, usage limits, and account health before running.

In [ ]:
from key_manager import ApifyKeyManager

km = ApifyKeyManager(tokens=APIFY_TOKENS, strategy="sequential")
km.print_summary()

## 5. Inspect Pending Venues (Resumability Check)
Checks how many total venues exist in the source database, how many have already been processed in `google_maps.db`, and how many remain pending.

In [ ]:
import sqlite3
import pandas as pd
from run_crawler import get_pending_cafes, init_output_db

if not os.path.exists(SOURCE_DB_PATH):
    print(f"⚠️ Notice: Source database not found at '{SOURCE_DB_PATH}'.")
    print("Please upload your 'cafes_cafe_shard00.db' to the Google Drive folder:")
    print(f"  -> {DRIVE_DIR}/")
else:
    init_output_db(OUTPUT_DB_PATH)
    pending = get_pending_cafes(source_db_path=SOURCE_DB_PATH, output_db_path=OUTPUT_DB_PATH)
    print(f"📊 Total Pending Venues to Scrape: {len(pending)}")
    
    if pending:
        df_sample = pd.DataFrame(pending[:5])
        print("\nPreview of Next 5 Pending Venues:")
        display(df_sample[["cafe_id", "name", "latitude", "longitude", "city"]])

## 6. Run Google Maps Scraper
Executes the batched scraper. Progress is automatically committed to SQLite and JSONL after every batch.

- If you want to run a test on a small slice, set `max_total=10` or `max_total=50`.
- Set `max_total=None` to scrape all remaining venues until completion.

In [ ]:
from run_crawler import run_scraper

# Set max_total=None for full run, or an integer for a limited test run
MAX_RUN_TOTAL = None  

crawl_stats = run_scraper(
    source_db_path=SOURCE_DB_PATH,
    output_db_path=OUTPUT_DB_PATH,
    raw_jsonl_path=RAW_JSONL_PATH,
    batch_size=BATCH_SIZE,
    max_total=MAX_RUN_TOTAL,
    max_images=MAX_IMAGES_PER_PLACE,
    max_reviews=MAX_REVIEWS_PER_PLACE,
    key_manager_instance=km,
    progress_bar=True
)

print("\nRun Session Stats:", crawl_stats)

## 7. Scraped Data Analytics & Verification
Query the SQLite database directly on Google Drive to view summary statistics and preview recent records.

In [ ]:
import sqlite3
import pandas as pd

if os.path.exists(OUTPUT_DB_PATH):
    conn = sqlite3.connect(OUTPUT_DB_PATH)
    
    print("--- Status Distribution ---")
    df_status = pd.read_sql_query("SELECT status, count(*) as count FROM places GROUP BY status", conn)
    display(df_status)
    
    print("\n--- Table Counts ---")
    for table in ["places", "place_images", "place_reviews", "place_attributes", "place_opening_hours"]:
        count = conn.execute(f"SELECT count(*) FROM {table}").fetchone()[0]
        print(f"  • {table:22}: {count:,} rows")
        
    print("\n--- Top Rated Cafes ---")
    df_top = pd.read_sql_query(
        "SELECT cafe_id, title, category_name, total_score, reviews_count, images_count, city "
        "FROM places WHERE status = 'success' AND reviews_count > 50 "
        "ORDER BY total_score DESC LIMIT 5",
        conn
    )
    display(df_top)
    
    conn.close()
else:
    print(f"Database not found at '{OUTPUT_DB_PATH}' yet.")

## 8. Download High-Resolution Photos (`=s0-rw` WebP)
Downloads camera-master resolution WebP photos directly into Google Drive.

- **Resumable:** Database download statuses are committed periodically (every 100 images). If stopped, it picks up right where it left off.
- `only_owner=True`: Downloads official cover photos and owner-uploaded images (cleanest quality).
- Set `only_owner=False` to download all photos including customer review attachments.

In [ ]:
from download_images import download_images

download_stats = download_images(
    output_db_path=OUTPUT_DB_PATH,
    images_dir=IMAGES_DIR,
    only_owner=True,           # True: owner photos only, False: all photos
    max_workers=MAX_IMAGE_WORKERS,
    batch_commit_size=100,
    progress_bar=True
)

print("\nDownload Session Summary:", download_stats)

## 9. Media & Storage Verification
Inspect the total disk space consumed by photos on Google Drive and display sample downloaded photos.

In [ ]:
import glob
from IPython.display import Image, display

if os.path.exists(IMAGES_DIR):
    image_files = glob.glob(os.path.join(IMAGES_DIR, "*", "*.webp"))
    total_bytes = sum(os.path.getsize(f) for f in image_files)
    mb_size = total_bytes / (1024 * 1024)
    
    print(f"📸 Total Photos Downloaded: {len(image_files):,}")
    print(f"💾 Total Disk Space Used:    {mb_size:.2f} MB ({mb_size / 1024:.2f} GB)")
    
    if image_files:
        print("\nPreview of 3 Downloaded Photos:")
        for img_path in image_files[:3]:
            print(f"  Image: {img_path} ({os.path.getsize(img_path) / 1024:.1f} KB)")
            display(Image(filename=img_path, width=400))
else:
    print("No images directory found yet.")